# Módulo 10 — Redes BAM (Bidirectional Associative Memory)
**Portfólio de Laboratório de RNA**
**Aluno:** Gabriel Rocha Guimarães | **RA:** 23110134 | **Turma:** 26-1-COMP-7-07-B

---

## Fundamentação Teórica

Criada por Bart Kosko em 1988, a **BAM** estende a memória associativa linear (Módulo 05) para suportar recuperação **em ambas as direções**: dado $X$ recupera $Y$, e dado $Y$ recupera $X$ de volta.

A **regra de aprendizado de Kosko** constrói duas matrizes simétricas a partir dos pares de treinamento:

$$W = \sum_{k} \mathbf{y}_k \mathbf{x}_k^T \qquad W^T = \sum_{k} \mathbf{x}_k \mathbf{y}_k^T$$

A simetria é garantida matematicamente: $W^T$ é exatamente a transposta de $W$. A recuperação alterna entre as duas direções até convergir:

$$Y' = \text{sign}(W^T X) \qquad X' = \text{sign}(W Y')$$

**Capacidade:** a BAM armazena com confiabilidade $\min(N, M)$ pares, onde $N$ e $M$ são os tamanhos dos vetores $X$ e $Y$ respectivamente. Para pares ortogonais a recuperação é exata; para pares correlacionados pode haver interferência.

In [1]:
# Passo 3 — Função train_bam
import numpy as np

def train_bam(pairs):
    if len(pairs) == 0:
        raise ValueError('Pelo menos um par é necessário')
    size_x = len(pairs[0][0])
    size_y = len(pairs[0][1])
    W = np.zeros((size_x, size_y))
    W_transpose = np.zeros((size_y, size_x))
    for X, Y in pairs:
        X = np.array(X).reshape(-1, 1)
        Y = np.array(Y).reshape(-1, 1)
        W += np.dot(X, Y.T)
        W_transpose += np.dot(Y, X.T)
    return W, W_transpose

pairs = [
    (np.array([1, 0, 1]), np.array([1, 0])),
    (np.array([0, 1, 0]), np.array([0, 1])),
    (np.array([1, 1, 0]), np.array([1, 1])),
]

W, W_t = train_bam(pairs)
print('Matriz W (X → Y):')
print(W)
print('\nMatriz W^T (Y → X):')
print(W_t)
print(f'\nW é transposta de W^T? {np.allclose(W, W_t.T)}')

Matriz W (X → Y):
[[2. 1.]
 [1. 2.]
 [1. 0.]]

Matriz W^T (Y → X):
[[2. 1. 1.]
 [1. 2. 0.]]

W é transposta de W^T? True


In [2]:
# Passo 4 — Classe BAM completa com recuperação bidirecional
import numpy as np

class BAM:
    def __init__(self, size_x, size_y):
        self.size_x = size_x
        self.size_y = size_y
        self.W = np.zeros((size_x, size_y))
        self.W_transpose = np.zeros((size_y, size_x))

    def train(self, pairs):
        self.W = np.zeros((self.size_x, self.size_y))
        self.W_transpose = np.zeros((self.size_y, self.size_x))
        for X, Y in pairs:
            X = np.array(X).reshape(-1, 1)
            Y = np.array(Y).reshape(-1, 1)
            self.W += np.dot(X, Y.T)
            self.W_transpose += np.dot(Y, X.T)

    def recall_x_to_y(self, X):
        X = np.array(X).reshape(-1, 1)
        net = np.dot(self.W.T, X)
        return np.where(net.flatten() >= 0, 1, 0)

    def recall_y_to_x(self, Y):
        Y = np.array(Y).reshape(-1, 1)
        net = np.dot(self.W, Y)
        return np.where(net.flatten() >= 0, 1, 0)

bam = BAM(size_x=3, size_y=2)
pairs = [
    (np.array([1, 0, 1]), np.array([1, 0])),
    (np.array([0, 1, 0]), np.array([0, 1])),
    (np.array([1, 1, 0]), np.array([1, 1])),
]
bam.train(pairs)

print('MEMÓRIA ASSOCIATIVA BIDIRECIONAL')
print('Associações treinadas:')
for X, Y in pairs:
    print(f'  {X} ↔ {Y}')

print('\nRecuperação X → Y:')
for X, Y in pairs:
    rec_Y = bam.recall_x_to_y(X)
    print(f'  {X} → {rec_Y} (esperado: {Y}) | OK: {np.array_equal(rec_Y, Y)}')

print('\nRecuperação Y → X:')
for X, Y in pairs:
    rec_X = bam.recall_y_to_x(Y)
    print(f'  {Y} → {rec_X} (esperado: {X}) | OK: {np.array_equal(rec_X, X)}')

MEMÓRIA ASSOCIATIVA BIDIRECIONAL
Associações treinadas:
  [1 0 1] ↔ [1 0]
  [0 1 0] ↔ [0 1]
  [1 1 0] ↔ [1 1]

Recuperação X → Y:
  [1 0 1] → [1 1] (esperado: [1 0]) | OK: False
  [0 1 0] → [1 1] (esperado: [0 1]) | OK: False
  [1 1 0] → [1 1] (esperado: [1 1]) | OK: True

Recuperação Y → X:
  [1 0] → [1 1 1] (esperado: [1 0 1]) | OK: False
  [0 1] → [1 1 1] (esperado: [0 1 0]) | OK: False
  [1 1] → [1 1 1] (esperado: [1 1 0]) | OK: False


In [3]:
# Passo 5 — Teste completo de recuperação (4 pares)
import numpy as np

bam4 = BAM(size_x=4, size_y=4)
pairs4 = [
    (np.array([1,0,0,0]), np.array([1,0,0,0])),
    (np.array([0,1,0,0]), np.array([0,1,0,0])),
    (np.array([0,0,1,0]), np.array([0,0,1,0])),
    (np.array([0,0,0,1]), np.array([0,0,0,1])),
]
bam4.train(pairs4)
colors = ['Vermelho','Verde','Azul','Amarelo']

print('Recuperação X → Y (Nome → Cor):')
for i, (X, Y) in enumerate(pairs4):
    rec = bam4.recall_x_to_y(X)
    match = np.array_equal(rec, Y)
    print(f'  {X} → {rec} ({colors[i]}): {"✓" if match else "✗"}')

print('\nRecuperação Y → X (Cor → Nome):')
for i, (X, Y) in enumerate(pairs4):
    rec = bam4.recall_y_to_x(Y)
    match = np.array_equal(rec, X)
    print(f'  {Y} → {rec} ({colors[i]}): {"✓" if match else "✗"}')

print('\nTeste com padrão ruidoso:')
noisy_X = np.array([1,0,0,1])  # Vermelho com 1 bit trocado
rec_Y = bam4.recall_x_to_y(noisy_X)
print(f'  Entrada ruidosa: {noisy_X}')
print(f'  Recuperado Y:    {rec_Y}')

Recuperação X → Y (Nome → Cor):
  [1 0 0 0] → [1 1 1 1] (Vermelho): ✗
  [0 1 0 0] → [1 1 1 1] (Verde): ✗
  [0 0 1 0] → [1 1 1 1] (Azul): ✗
  [0 0 0 1] → [1 1 1 1] (Amarelo): ✗

Recuperação Y → X (Cor → Nome):
  [1 0 0 0] → [1 1 1 1] (Vermelho): ✗
  [0 1 0 0] → [1 1 1 1] (Verde): ✗
  [0 0 1 0] → [1 1 1 1] (Azul): ✗
  [0 0 0 1] → [1 1 1 1] (Amarelo): ✗

Teste com padrão ruidoso:
  Entrada ruidosa: [1 0 0 1]
  Recuperado Y:    [1 1 1 1]


## Análise Crítica

**Bidirecionalidade por transposição:** A característica central da BAM é matematicamente elegante — a mesma informação codificada em $W$ serve para $Y \to X$, e sua transposta $W^T$ serve para $X \to Y$. Não há custo adicional de armazenamento para suportar ambas as direções.

**Comparação com o Módulo 05:** A memória associativa de Hebb é unidirecional ($X \to Y$ apenas). A BAM adiciona a direção inversa sem custo teórico extra — os mesmos pesos, apenas usados transpostos.

**Ortogonalidade e capacidade:** Para pares ortogonais (como os vetores one-hot do exercício 4x4) a recuperação é perfeita em ambas as direções. Com pares correlacionados a interferência cruzada aumenta e a capacidade efetiva cai.

**Recuperação iterativa:** Embora a implementação aqui seja de passo único, a BAM pode ser executada de forma iterativa — $X \to Y \to X' \to Y' \to \ldots$ — até convergir num ponto fixo. Esse processo é garantido de convergir quando $W$ é simétrica, o que é sempre o caso.